In [2]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os

In [3]:
def remove_single_interactions(df):
    while True:
        count_users = df["id_user"].value_counts(sort=False)
        count_items = df["id_item"].value_counts(sort=False)
        invalid_users = count_users[count_users==1].index
        invalid_items = count_items[count_items==1].index
        if len(invalid_users) == 0 and len(invalid_items) == 0:
            break
        df = df[(~df["id_user"].isin(invalid_users))&(~df["id_item"].isin(invalid_items))].copy()
    return df

def timestamp_diff(df, threshold):

    #Calcula a diferença de tempo entre a iteração atual e a passada, por usuário
    def calc_diff(df_group):

        df_group = df_group.sort_values('datetime')
        df_group['time_diff'] = df_group['datetime'] - df_group['datetime'].shift(1)
        df_group['time_diff'] = df_group['time_diff'].fillna(pd.to_timedelta(0, unit='s'))
        df_group['time_diff'] = df_group['time_diff'].astype('int64')/ 10**9

        non_noise_diffs = df_group[df_group['time_diff'] > threshold]
        df_group['Q1'] = non_noise_diffs['time_diff'].quantile(0.25)
        df_group['Q3'] = non_noise_diffs['time_diff'].quantile(0.75)
        #IQR = Q3 - Q1

        return non_noise_diffs

    if 'timestamp' in df.columns:
        df['datetime'] = pd.to_datetime(df['timestamp'], unit='s')
    elif 'datetime' in df.columns:
        df['datetime'] = pd.to_datetime(df['datetime']).dt.floor('s')

    # Gera a coluna de diferença entre iterações
    df = df.groupby('id_user', group_keys=False).apply(calc_diff).reset_index(drop=True)
    
    return df

def user_item_distribution(user_interactions, item_interactions):
    
    user_item = [user_interactions, item_interactions]
    
    # Create a subplot figure
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("User Interaction Distribution", "Item Interaction Distribution")
    )
    
    for i in range(len(user_item)):
        
        user_interactions = user_item[i]
        
        user_bins = pd.cut(
            user_interactions,
            bins=[0, 5, 10, 15, float("inf")],
            labels=["1-5", "6-10", "10-15", "16+"]
        )

        user_distribution = user_bins.value_counts().sort_index()

        # Add User Distribution Plot
        fig.add_trace(
            go.Bar(
                x=user_distribution.index.astype(str),
                y=user_distribution.values,
                text=user_distribution.values,
                textposition='outside',
                name="Users"
            ),
            row=1, col=i+1
        )

    # Update layout for both plots
    fig.update_layout(
        showlegend=False,
        template="plotly_white",
        height=450,
        width=700 
    )

    fig.update_xaxes(title_text="Intervalo", row=1, col=1)
    fig.update_xaxes(title_text="Intervalo", row=1, col=2)
    fig.update_yaxes(title_text="Número de usuários", row=1, col=1)
    fig.update_yaxes(title_text="Número de itens", row=1, col=2)

    fig.show()
    
def time_boxplot(data):
    # Create a box plot using Plotly
    fig = px.box(data, y="time_diff", log_y=True)

    q1 = data["time_diff"].quantile(0.25)
    q3 = data["time_diff"].quantile(0.75)
    iqr = q3 - q1

    upper_bound = q3 + 1.5 * iqr
    outliers = data["time_diff"][data["time_diff"] > upper_bound]
    outliers_q3 = data["time_diff"][data["time_diff"] > q3]
    outlier_count = len(outliers)
    outliers_q3_count = len(outliers_q3)

    # Customize layout
    fig.update_layout(
        title="Box Plot do intervalo de tempo - " + f"Valores acima de Q3: {outliers_q3_count}, outliers: {outlier_count}",
        yaxis_title="Intervalo de tempo (segundos)",
        template="plotly_white",
        height=400,
        width=700
    )

    # Show the plot
    fig.show()

In [4]:
df_ml = pd.read_csv('datasets/MovieLens/interactions.csv', sep=";", header='infer')
df_ml = remove_single_interactions(df_ml)

df_rt = pd.read_csv('datasets/RetailRocket-Transactions/interactions.csv', sep=";", header='infer')
df_rt = remove_single_interactions(df_rt)

df_db = pd.read_csv('datasets/DeliciousBookmarks/interactions.csv', sep=";", header='infer')
df_db = remove_single_interactions(df_db)

In [ ]:
datasets = ['Taobao']

for dataset in datasets:
    
    data = pd.read_csv(os.path.join("datasets", dataset, "interactions.csv"), sep=";", header='infer')
    data = remove_single_interactions(data)

    print("==================//===================")
    print(dataset)
    print("==================//===================\n")
    
    print("Quantidade de interações na base", data.shape[0], '\n')

    print("Usuários únicos:", data["id_user"].nunique())
    user_interactions = data.groupby("id_user").size()
    print("Média de interações por usuário:", round(user_interactions.mean(), 2))
    print("Maximo:", round(user_interactions.max(), 2), '\n')

    print("Itens únicos:", data["id_item"].nunique())
    item_interactions = data.groupby("id_item").size()
    print("Média de interações por item:", round(item_interactions.mean(), 2))

    user_item_distribution(user_interactions, item_interactions)
    data = timestamp_diff(data, 300)
    time_boxplot(data)

==================//===================
Taobao
==================//===================

Quantidade de interações na base 117088 

Usuários únicos: 14637
Média de interações por usuário: 8.0
Maximo: 103 

Itens únicos: 26503
Média de interações por item: 4.42


In [ ]:
# Simulating a DataFrame with time intervals (replace this with your actual data)
data = timestamp_diff(df_db, 0)



In [6]:
df = pd.read_csv('datasets/RetailRocket-Transactions/interactions.csv', sep=";", header='infer')

In [7]:
print(df)

                      datetime  id_user  id_item
0      2015-06-02 05:17:56.276   599528   356475
1      2015-06-01 21:18:20.981   121688    15335
2      2015-06-01 21:25:15.008   552148    81345
3      2015-06-01 16:38:56.375   102019   150318
4      2015-06-01 16:01:58.180   189384   310791
...                        ...      ...      ...
22452  2015-07-31 21:12:56.570  1050575    31640
22453  2015-07-31 21:57:58.779   861299   456602
22454  2015-07-31 15:48:50.123   855941   235771
22455  2015-07-31 15:12:40.300   548772    29167
22456  2015-07-31 16:09:49.163  1051054   312728

[22457 rows x 3 columns]


In [8]:
from sklearn.model_selection import train_test_split

user_item_count = df.groupby('id_user').size()
data = df[df['id_user'].isin(user_item_count[user_item_count >= 5].index)]
data = data.sort_values(by=["id_user", "datetime"])

df_train = data.loc[data.groupby('id_user').apply(lambda x: x.index[:-2]).explode()]
df_test_aux = data.loc[data.groupby('id_user').apply(lambda x: x.index[-1])]
df_val, df_test = train_test_split(data1.sort_values('datetime'), test_size=0.8, random_state=42)

NameError: name 'data1' is not defined

In [7]:
df = pd.read_csv('datasets/BestBuy/interactions_full.csv', sep=";", header='infer')
df = df[['id_user', 'id_item', 'datetime']]

In [15]:
percentage_to_keep = 0.3
unique_users = df['id_user'].unique()
num_users_to_remove = int(len(unique_users) * (1-percentage_to_keep))
users_to_remove = np.random.choice(unique_users, num_users_to_remove, replace=False)
df_filtered = df[~df['id_user'].isin(users_to_remove)]

                                          id_user  id_item  \
0        000000df17cd56a5df4a94074e133c9d4739fae3  2125233   
5        00001ff8394a2d9bd7adffb1547180bf5bbfc4e5  2416092   
8        00003cb3f85244c652f22c1daf11aed35d5ab7f6  8280834   
9        0000404374364b75c80d384c2a8c1d379237a2d9  2740208   
11       00006d8bb4bcc66f1676237629968277fb7a8dc5  1230537   
...                                           ...      ...   
1865250  fffea2ad9d576bba80cb2771fde4fde285bda2f2  2945413   
1865256  fffef9353f39b1e79ace581ed5893d1960856f4c  1246273   
1865259  ffff784f7af65f8618e8255d2b04f8028626730f  2025119   
1865264  ffffbcfcee8ff636231fa0df557528e6fa2ecdba  3674224   
1865265  ffffddbbdc67c4d775c7232082adaf40c7e7d03c  2986037   

                    datetime  
0        2011-09-01 23:44:52  
5        2011-09-07 15:54:47  
8        2011-08-29 13:37:32  
9        2011-10-25 22:55:58  
11       2011-10-18 17:21:33  
...                      ...  
1865250  2011-09-20 20:11:28  
1865256